# ACE-Step 1.5 — Generador de musica Lofi para LofiProducer

**Licencia MIT** | **GPU T4 gratuita** | **Instrumental**

Ejecutar todo con `Runtime → Run all` y descargar los WAV generados.

In [ ]:
# 1. Instalar ACE-Step 1.5
!git clone https://github.com/ace-step/ACE-Step.git
%cd ACE-Step
!pip install -e . 2>&1 | tail -5

In [ ]:
# 2. Configurar para GPU T4 (float32 + sin LM)
import torch
assert torch.cuda.is_available(), 'GPU no disponible'
print('GPU:', torch.cuda.get_device_name(0))
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
%env PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True

In [ ]:
# 3. Generar 15 pistas de Lofi
from acestep.pipeline import ACEStepPipeline
import os

pipe = ACEStepPipeline(
    checkpoint_dir='./checkpoints',
    device='cuda',
    dtype='float32',
    torch_compile=False,
    cpu_offload=True,
    use_lm=False,          # Desactivado: ahorra 1.7GB VRAM
    use_mp3=False,
    use_suno_style=False,
)

prompts = [
    'lofi hip hop, 80 BPM, warm Rhodes piano, vinyl crackle, rain ambiance, no vocals',
    'chillhop beat, 75 BPM, dusty drums, jazzy chords, mellow bass, instrumental',
    'jazzhop instrumental, 85 BPM, upright bass, brushed drums, soft piano, night mood',
    'ambient lofi, 70 BPM, atmospheric pads, distant thunder, gentle piano, no drums',
    'bedroom pop lofi, 90 BPM, dreamy guitar, soft synths, tape saturation, calm vibe',
    'late night jazzhop, 78 BPM, muted trumpet, warm electric piano, walking bass',
    'rainy day lofi, 72 BPM, deep sub bass, tape hiss, melancholic piano, instrumental',
    'study beats, 82 BPM, mellow piano loop, hip hop drums, vinyl warmth, focus',
    'coffee shop lofi, 88 BPM, light percussion, wurlitzer, acoustic bass, relaxed',
    'midnight chillhop, 76 BPM, ambient guitar, soft drums, deep atmosphere, no vocals',
    'smooth jazz lofi, 84 BPM, saxophone, keys, shuffle drums, evening vibe',
    'rainy window beats, 74 BPM, filtered pads, tape delay, mellow rhythm, chill',
    'sunset lofi, 79 BPM, warm analog synth, laid-back drums, mellow bass, instrumental',
    'aesthetic lofi, 81 BPM, kalimba, soft kick, dreamy reverb, vinyl noise, no vocals',
    'focus flow lofi, 86 BPM, minimal piano, gentle percussion, ambient texture',
]

os.makedirs('outputs', exist_ok=True)
for i, prompt in enumerate(prompts, 1):
    print(f'\n[{i}/{len(prompts)}] {prompt[:60]}...')
    try:
        audio = pipe(
            format='wav',
            audio_duration=90,        # 90 segundos por pista
            prompt=prompt,
            lyrics='[instrumental]',
            infer_step=15,
            guidance_scale=7.5,
            scheduler_type='euler',
            cfg_type='apg',
            omega_scale=10.0,
            manual_seeds=[42 + i],
            guidance_interval=0.5,
            guidance_interval_decay=0.0,
            min_guidance_scale=3.0,
            use_erg_tag=True,
            use_erg_lyric=False,
            use_erg_diffusion=True,
            oss_steps=None,
            guidance_scale_text=0.0,
            save_path=f'outputs/track_{i:02d}.wav',
        )
        print(f'  OK -> outputs/track_{i:02d}.wav')
    except Exception as e:
        print(f'  ERROR: {e}')

In [ ]:
# 4. Empaquetar en ZIP para descarga
!zip -r lofi_tracks.zip outputs/
print('Descargar lofi_tracks.zip desde el panel de archivos (icono carpeta)')